# VoxTell on Neurite-OASIS -- PLAN ROWS 12-16: image flip, organ noise, embedding intervention, attention, layer distance (Kaggle GPU)

The five experiments the earlier notebooks could not run (they only used the public API and final masks):

| Row | Experiment | Question |
|---|---|---|
| 12 | Flip the image left-right, keep the prompt | Does "left" follow the patient's anatomy, or the left side of the image? |
| 13 | Fill the organ / both organs with noise, or use a noise image | Does the model need image evidence, or does it draw a memorized organ? |
| 14 | Blend or patch the "left" and "right" representations inside the model | Where is the side decided, and does the mask change smoothly? |
| 15 | Cross-attention of each prompt over the organ regions | Does "left kidney" look at the left kidney inside the model? |
| 16 | Distance between the left and right query states at every decoder layer | At which layer do left and right first differ? |

- Subjects: **10** (OASIS_OAS1_0014_MR1, OASIS_OAS1_0057_MR1, OASIS_OAS1_0123_MR1, OASIS_OAS1_0135_MR1, OASIS_OAS1_0062_MR1, OASIS_OAS1_0363_MR1, OASIS_OAS1_0419_MR1, OASIS_OAS1_0076_MR1, OASIS_OAS1_0151_MR1, OASIS_OAS1_0421_MR1); organs: **cerebral white matter, cerebellum cortex**; no rotation.
- Rows 14-16 use one custom sliding-window pass per subject (`voxtell_mechanistic.run_outputs`): a single image-encoder
  pass per patch serves every variant. The custom loop is **checked against the predictor's own output** before anything
  else runs (section 3), and the "whole state" patch of a layer is a built-in positive control (it must reproduce the
  source prompt's mask, up to rounding).
- Output: CSVs under `results/` (+ `attention_maps/`), zipped at the end.
- To run only part of the notebook, skip the sections you do not need; each section is independent once sections 1-3 have run.


In [ ]:
import subprocess, sys, re
from pathlib import Path

# This cell installs everything (voxtell + deps, a pinned transformers, and a clean
# torch+torchvision pair) BEFORE torch is ever imported in this kernel, then requires a
# kernel restart. Two things force this ordering:
#
# 1. `pip install voxtell` pins torch<2.9, so it can silently downgrade Kaggle's preinstalled
#    torch (observed both outcomes across runs -- sometimes torch stays put, sometimes it's
#    pulled to 2.8.x) as a side effect of installing voxtell itself.
# 2. Separately, Kaggle's preinstalled torch build has been observed shipping in a
#    self-inconsistent state regardless: Python files referencing C-level symbols (e.g.
#    torch.distributed.tensor._ops._embedding_ops._MaskPartial) that the paired compiled
#    extension doesn't actually provide -- surfaces only when importing torch's heavier
#    submodules (dynamo/FSDP), which VoxTell's predictor does via
#    `from torch._dynamo import OptimizedModule`.
#
# Both problems need a clean, paired reinstall of torch+torchvision (one pip command, no
# --no-deps, so pip picks a mutually ABI-compatible pair -- an earlier version of this fix
# reinstalled torch alone, which left the already-installed torchvision mismatched and broke
# with "RuntimeError: operator torchvision::nms does not exist"). But torch's C extension
# (`torch._C`) is a process-wide singleton: once loaded, Python can't safely swap its files
# out from under itself (tried purging sys.modules and re-importing in the same kernel --
# raises "RuntimeError: function '_has_torch_function' already has a docstring"). So this
# entire install+fix sequence has to run, and finish, WITHOUT importing torch anywhere in
# this kernel -- read the installed version via `pip show` (subprocess), never `import
# torch` -- and the kernel must be restarted afterward before anything imports torch.
_marker = Path('/kaggle/working/.env_setup_done')
if _marker.exists():
    print('environment already installed/fixed earlier in this session -- skipping. '
          f'({_marker.read_text()})')
else:
    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

    print('installing voxtell + deps...')
    _pip('voxtell', 'nibabel', 'pandas', 'matplotlib', 'huggingface_hub')

    _show = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                            capture_output=True, text=True, check=True).stdout
    _torch_version = re.search(r'^Version: (.+)$', _show, re.MULTILINE).group(1).split('+')[0]
    print(f'reinstalling torch=={_torch_version} + torchvision cleanly (this can take a minute)...')
    _pip('--force-reinstall', '--no-cache-dir', f'torch=={_torch_version}', 'torchvision')

    # transformers ALONE (no other top-level packages in the same command, no --no-deps) so
    # pip resolves its own tokenizers/huggingface_hub requirements together with it -- e.g.
    # transformers 4.57.6 needs huggingface-hub<1.0,>=0.34.0 and tokenizers<=0.23.0,>=0.22.0.
    # Bundling unconstrained tokenizers/huggingface_hub upgrades into a separate command (an
    # earlier version of this fix did that) pulled newer versions that then violated
    # transformers's own pins.
    print('reinstalling transformers pinned <5...')
    _pip('--force-reinstall', '--no-cache-dir', 'transformers>=4.54.0,<5')

    _marker.write_text(f'torch=={_torch_version}')
    raise RuntimeError(
        'Environment installed/fixed. RESTART THE KERNEL NOW (Kaggle: Run -> Restart '
        'session, or the restart icon in the toolbar), then re-run this notebook from the '
        'top -- this cell will then just print "skipping" and the rest of the notebook will '
        'run against the clean, consistent install. Do not run any later cell before '
        'restarting: torch would still be broken/mismatched in THIS kernel process even '
        'though the files on disk are now fixed.'
    )


## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU visible -- enable GPU under Notebook Settings -> Accelerator."
print("CUDA OK:", torch.cuda.get_device_name(0))


## 2. Setup Working Directories (Kaggle Environment)

In [ ]:
import os
from pathlib import Path

# /kaggle/working is the writable output directory that persists in Kaggle outputs
WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT)


## 3. Install VoxTell

VoxTell is pip-installable directly (no cloning/build-from-source, no pinned-numpy fights
like Medical-SAM3 needed). PyTorch is already present in the Kaggle GPU image, so we install
VoxTell without re-pulling torch.

In [ ]:
%cd /kaggle/working

# All installation/reinstallation (voxtell + deps, transformers pin, the torch+torchvision
# consistency fix) happens in the first cell of this notebook, BEFORE torch is ever imported
# in this kernel -- see that cell for why. This cell just verifies the result.
import torch
import transformers
import voxtell
print('transformers version:', transformers.__version__)
assert int(transformers.__version__.split('.')[0]) < 5,     f"transformers {transformers.__version__} violates voxtell's own transformers<5 pin -- re-run the first cell and restart the kernel."
print('voxtell installed:', voxtell.__file__)
print('torch version:', torch.__version__, '-- CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available -- check the first cell's pip output above."


## 4. Clone your research repo (Neurite OASIS volumes/manifest)

Uses a Kaggle Secret instead of a hardcoded token — this cell never has a real token written
into the notebook file itself.

In [ ]:
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT (Add-ons -> Secrets) before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'faria/voxtell3d-rotation-medical-segmentation', '--depth', '1', _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    raise RuntimeError(f'git clone failed (exit {_result.returncode}):\n{_stderr_scrubbed}')

RESEARCH_REPO = Path(_dest)
print('cloned into', RESEARCH_REPO)
print('repo root:', os.listdir(RESEARCH_REPO))
print('annotator folders present:', [p for p in ('faria', 'nipu', 'hridoy') if (RESEARCH_REPO / p).is_dir()])


## 5. Download and load the VoxTell model

**Correction:** the published PyPI package (`voxtell`, v0.1.0 -- confirmed by checking the
actual tagged source on GitHub, since `pip install voxtell` does not track the `main` branch)
requires `model_dir` as a **required** argument to `VoxTellPredictor` -- there is no
auto-download-on-first-use in this released version (that convenience helper exists only on
GitHub's `main` branch, not yet in a PyPI release). So we download the checkpoint ourselves
first, the same way Medical-SAM3's notebook does its checkpoint fetch, then pass the resulting
local folder in as `model_dir`.

Checkpoint: [huggingface.co/mrokuss/VoxTell](https://huggingface.co/mrokuss/VoxTell), folder
`voxtell_v1.1` (contains `plans.json` and `fold_0/checkpoint_final.pth`).

In [ ]:
import torch
from huggingface_hub import snapshot_download
from voxtell.inference.predictor import VoxTellPredictor

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
CKPT_CACHE_DIR = WORKING_DIR / 'checkpoints'
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = str(Path(_local_repo_dir) / VOXTELL_MODEL_VERSION)
print('VoxTell checkpoint at', VOXTELL_MODEL_DIR)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
predictor = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
print('VoxTell predictor ready on', device)


### 5b. Fix: text encoder OOM on a single T4/P100

VoxTell's text encoder is `Qwen/Qwen3-Embedding-4B` (4 billion parameters). The installed
v0.1.0 release loads it with `AutoModel.from_pretrained(text_encoding_model)`, no `torch_dtype`
given -- Hugging Face's default in that case is **full fp32**, so the text encoder alone is
~16GB, larger than a T4's entire 14.56GB usable memory, before the segmentation network even
factors in. This is why the smoke test OOM'd on `self.text_backbone.to(self.device)` --
it has nothing to do with passing 13 organs at once (the model dominates the memory cost, not
the number of prompts).

There's no public constructor argument to fix this, so this cell swaps in a half-precision
copy of the text encoder after the predictor is built (roughly halving its footprint to ~8GB,
which should coexist with the segmentation network), and wraps `embed_text_prompts` so its
output is always cast back to `float32` before being used further -- avoiding any fp16/fp32
dtype-mismatch risk in the segmentation network's fusion layers downstream. It reuses VoxTell's
own original `embed_text_prompts` method internally (just swaps the model and casts the
result), so none of its internal tokenization/pooling logic is duplicated here.

In [ ]:
import torch
from transformers import AutoModel

# Guarded so re-running this cell never re-wraps an already-wrapped method --
# the true original is captured on the predictor object itself, exactly once.
if not hasattr(predictor, '_orig_embed_text_prompts'):
    predictor._orig_embed_text_prompts = predictor.embed_text_prompts

predictor.text_backbone = AutoModel.from_pretrained(
    'Qwen/Qwen3-Embedding-4B', dtype=torch.float16  # VoxTell v0.1.0's default text encoder
).eval()

@torch.inference_mode()
def _embed_text_prompts_fp16_safe(text_prompts):
    embeddings = predictor._orig_embed_text_prompts(text_prompts)
    return embeddings.float()

predictor.embed_text_prompts = _embed_text_prompts_fp16_safe
print('text encoder swapped to fp16 (output still cast back to float32)')


## 6. Helpers (Dice / centroid / bbox / side, same logic as scripts/common3d.py)

In [ ]:
import csv, time
from pathlib import Path

import numpy as np
import torch
from nnunetv2.imageio.nibabel_reader_writer import NibabelIOWithReorient
from acvl_utils.cropping_and_padding.bounding_boxes import insert_crop_into_image

RESEARCH_REPO = Path(RESEARCH_REPO)
reader = NibabelIOWithReorient()

_OPPOSITE = {"R": "L", "L": "R", "A": "P", "P": "A", "S": "I", "I": "S"}

def load_volume_ras(path):
    """Load a NIfTI image via VoxTell's own reader so predicted masks and ground
    truth end up in the exact same RAS-reoriented array space. Returns (1, X, Y, Z),
    the channel-first shape voxtell.inference.predictor expects."""
    data, props = reader.read_images([str(path)])
    return np.asarray(data), props

def spatial_view(arr_4d):
    a = np.asarray(arr_4d)
    while a.ndim > 3 and a.shape[0] == 1:
        a = a[0]
    return a

def load_seg_ras(path):
    data, props = reader.read_seg(str(path))
    return spatial_view(data).astype(np.int32), props

# NibabelIOWithReorient (nnU-Net's reader) returns arrays in REVERSED axis order
# relative to the canonical RAS affine -- i.e. axis0 is Superior-Inferior, axis1 is
# Anterior-Posterior, axis2 is Right-Left (not the naive (R,A,S) <-> (0,1,2) mapping).
# Verified empirically against real predictions: left/right kidney centroids differ by
# ~85 voxels on axis2 but only ~3 voxels on axis0, for every subject. Getting this wrong
# silently breaks every side/laterality computation (Dice is unaffected -- voxel overlap
# doesn't care about axis labels -- but side, centroid-shift, and causal-agreement do).
AXCODES = ("S", "A", "R")
RL_AXIS = AXCODES.index("R")

def mask_centroid_side(mask: np.ndarray):
    """Centroid + bbox + anatomical side in RAS-reoriented voxel space (see AXCODES
    above for the actual axis order). Returns (centroid, bbox, sides) where
    bbox = (min0,min1,min2,max0,max1,max2)."""
    coords = np.nonzero(mask)
    if len(coords[0]) == 0:
        return None
    centroid = tuple(float(c.mean()) for c in coords)
    bbox = tuple(int(c.min()) for c in coords) + tuple(int(c.max()) for c in coords)
    sides = tuple(code if centroid_val > mask.shape[axis] / 2 else _OPPOSITE[code]
                  for axis, (code, centroid_val) in enumerate(zip(AXCODES, centroid)))
    return centroid, bbox, sides

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def iou_score(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return 1.0 if union == 0 else float(inter / union)

def struct_slug(name: str) -> str:
    return name.strip().lower().replace(' ', '_').replace('/', '-')

def load_manifest_rot000(person, dataset_name):
    """Only rot000 rows -- this investigation doesn't use the rotation sweep."""
    path = RESEARCH_REPO / person / 'samples' / dataset_name / f'manifest_{dataset_name}.csv'
    with open(path, newline='') as f:
        return [row for row in csv.DictReader(f) if int(row['rotation_angle_deg']) == 0]

print('helpers ready')


import base64
import zlib


def encode_mask_rle(mask: np.ndarray) -> str:
    """Run-length-encode a boolean mask (C-order flatten), zlib-compress, base64-encode."""
    flat = np.asarray(mask, dtype=bool).ravel(order='C')
    change_idx = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    boundaries = np.concatenate(([0], change_idx, [flat.size]))
    runs = np.diff(boundaries).astype(np.uint32)
    if flat.size and flat[0]:
        runs = np.concatenate(([0], runs)).astype(np.uint32)
    raw = zlib.compress(runs.tobytes(), level=9)
    return base64.b64encode(raw).decode('ascii')


def decode_mask_rle(encoded: str, shape) -> np.ndarray:
    runs = np.frombuffer(zlib.decompress(base64.b64decode(encoded)), dtype=np.uint32)
    flat = np.zeros(int(np.prod(shape)), dtype=bool)
    idx = 0
    value = False
    for run_len in runs:
        if value and run_len:
            flat[idx:idx + run_len] = True
        idx += int(run_len)
        value = not value
    return flat.reshape(shape, order='C')


## 1. Load the experiment code

In [ ]:
import json, sys, types

# The experiment code is embedded here so the notebook is self-contained. (Sources: scripts/voxtell_mechanistic.py and
# scripts/voxtell_experiments.py in the repo -- edit them there and rebuild with scripts/build_mechanistic_notebooks.py.)
_MECH_SRC = json.loads('"\\"\\"\\"\\nMechanistic helpers for the VoxTell groundedness investigation (plan rows 12-15).\\n\\nEverything here is written against the installed `voxtell` v0.1.0 source\\n(voxtell/model/voxtell_model.py, voxtell/model/transformer.py,\\nvoxtell/inference/predictor.py) and reuses the model\'s own modules -- nothing in\\nvoxtell is monkeypatched.\\n\\nFacts about VoxTell this code depends on (all verified from the source):\\n  - The image encoder never sees the text; its layer-4 features (12x12x12 tokens\\n    for a 192^3 patch) become the cross-attention *memory*.\\n  - The text embedding is projected (`project_text_embed`) and becomes the initial\\n    query state of a 6-layer pre-norm transformer decoder (`normalize_before=True`\\n    => `forward_pre`: cross-attention + FFN only, NO self-attention between the\\n    queries, so prompts in a batch never interact).\\n  - Each layer returns (state, cross-attention weights averaged over heads,\\n    shape (1, n_queries, n_tokens)).\\n  - After the last layer a LayerNorm gives the \\"mask embedding\\", which the\\n    convolutional decoder turns into a mask (one full decoder pass per prompt).\\n  - Because the decoder is a residual stream, the original text only reaches later\\n    layers through the running state. Replacing the *whole state* at layer k with\\n    another prompt\'s state therefore reproduces that prompt exactly (used as a\\n    positive control); the informative intervention is to replace a single layer\'s\\n    *update* (state_out - state_in).\\n\\nPublic API\\n  tokens_to_patch_grid, encode, base_trajectory, patched_state, decode,\\n  AttnAccumulator, run_outputs, score_masks\\n\\"\\"\\"\\n\\nimport contextlib\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom einops import rearrange\\n\\n\\n# ---------------------------------------------------------------------------------\\n# token <-> voxel bookkeeping\\n# ---------------------------------------------------------------------------------\\ndef tokens_to_patch_grid(w, g):\\n    \\"\\"\\"`w`: (..., g**3) values over VoxTell\'s image tokens -> (..., g, g, g) indexed in\\n    the patch\'s own axis order (axis0, axis1, axis2).\\n\\n    VoxTell flattens tokens as `rearrange(feat, \'b c d h w -> (h w d) b c\')`, i.e.\\n    token index = (h*g + w)*g + d, where (d, h, w) are the patch axes (0, 1, 2).\\n    \\"\\"\\"\\n    lead = w.shape[:-1]\\n    x = w.reshape(*lead, g, g, g)  # [h, w, d] == [axis1, axis2, axis0]\\n    nd = x.ndim\\n    return x.permute(*range(nd - 3), nd - 1, nd - 3, nd - 2)  # -> [d, h, w]\\n\\n\\n# ---------------------------------------------------------------------------------\\n# model pieces (mirror VoxTellModel.forward, split so we can read / patch the decoder)\\n# ---------------------------------------------------------------------------------\\ndef encode(model, patch):\\n    \\"\\"\\"patch: (1, C, X, Y, Z) -> (skips, memory (S, 1, Q)).\\"\\"\\"\\n    skips = model.encoder(patch)\\n    feat = skips[model.selected_decoder_layer]\\n    mem = rearrange(feat, \'b c d h w -> b h w d c\')\\n    mem = model.project_bottleneck_embed(mem)\\n    mem = rearrange(mem, \'b h w d c -> (h w d) b c\')\\n    return skips, mem\\n\\n\\ndef base_trajectory(model, tgt0, memory, pos):\\n    \\"\\"\\"Run the decoder layers on `tgt0` (M, 1, Q). Returns (states, attns):\\n    states[k] is the state entering layer k (states[-1] = after the last layer, before\\n    the final norm); attns[k] = (1, M, S) cross-attention weights of layer k.\\"\\"\\"\\n    layers = model.transformer_decoder.layers\\n    assert all(l.normalize_before for l in layers), \'expected the pre-norm (forward_pre) decoder\'\\n    states, attns = [tgt0], []\\n    t = tgt0\\n    for layer in layers:\\n        t, w = layer(t, memory, pos=pos)\\n        states.append(t)\\n        attns.append(w)\\n    return states, attns\\n\\n\\ndef patched_state(model, tgt0_row, memory, pos, base_states, ops):\\n    \\"\\"\\"Re-run the decoder for ONE query (tgt0_row: (1, 1, Q)) with interventions.\\n\\n    ops: list of dicts {\'layer\': k, \'mode\': \'update\' | \'state\', \'src\': row}\\n      - \'update\': the layer-k output is `state_in + (src\'s layer-k update)`, i.e. only that\\n                  layer\'s contribution is taken from the source prompt\\n      - \'state\' : the layer-k output is the source prompt\'s whole state (positive control)\\n    `base_states` is the list returned by `base_trajectory` for all prompts (M rows).\\"\\"\\"\\n    by_layer = {}\\n    for op in ops:\\n        by_layer.setdefault(op[\'layer\'], []).append(op)\\n    t = tgt0_row\\n    for k, layer in enumerate(model.transformer_decoder.layers):\\n        new, _ = layer(t, memory, pos=pos)\\n        for op in by_layer.get(k, []):\\n            s = op[\'src\']\\n            if op[\'mode\'] == \'update\':\\n                new = t + (base_states[k + 1][s:s + 1] - base_states[k][s:s + 1])\\n            elif op[\'mode\'] == \'state\':\\n                new = base_states[k + 1][s:s + 1].clone()\\n            else:\\n                raise ValueError(op[\'mode\'])\\n        t = new\\n    return t\\n\\n\\ndef decode(model, skips, mask_emb):\\n    \\"\\"\\"mask_emb: (n, Q) -> logits (n, X, Y, Z). Same per-prompt loop as VoxTellModel.forward.\\"\\"\\"\\n    me = mask_emb.unsqueeze(0)  # (1, n, Q)\\n    mes = [proj(me) for proj in model.project_to_decoder_channels]\\n    outs = []\\n    for i in range(me.shape[1]):\\n        prompt_embeds = [m[:, i:i + 1] for m in mes]\\n        outs.append(model.decoder(skips, prompt_embeds)[0])\\n    return torch.cat(outs, dim=1)[0]\\n\\n\\n# ---------------------------------------------------------------------------------\\n# attention accumulation over the sliding window\\n# ---------------------------------------------------------------------------------\\nclass AttnAccumulator:\\n    \\"\\"\\"Accumulates cross-attention *density* (attention weight x n_tokens, so the mean is 1)\\n    into voxel space, weighted by the same Gaussian as the logits.\\"\\"\\"\\n\\n    def __init__(self, spatial_shape, n_layers, n_rows, patch_size, device):\\n        self.buf = torch.zeros((n_layers, n_rows, *spatial_shape), dtype=torch.float32, device=device)\\n        self.cnt = torch.zeros(tuple(spatial_shape), dtype=torch.float32, device=device)\\n        self.patch_size = tuple(int(p) for p in patch_size)\\n\\n    def add(self, slicer, attn, gaussian):\\n        \\"\\"\\"attn: (L, M, S) float; gaussian: patch-shaped weights.\\"\\"\\"\\n        L, M, S = attn.shape\\n        g = int(round(S ** (1.0 / 3.0)))\\n        assert g ** 3 == S, f\'{S} tokens is not a cube\'\\n        dens = tokens_to_patch_grid(attn.float(), g) * S  # (L, M, g, g, g)\\n        gz = gaussian.float()\\n        for l in range(L):\\n            up = F.interpolate(dens[l][None], size=self.patch_size, mode=\'nearest\')[0]  # (M, X, Y, Z)\\n            self.buf[(l, slice(None), *slicer[1:])] += up * gz\\n        self.cnt[slicer[1:]] += gz\\n\\n    def finalize(self, revert_slicer):\\n        out = self.buf / self.cnt.clamp_min(1e-6)\\n        return out[(slice(None), slice(None), *revert_slicer[1:])]\\n\\n\\n# ---------------------------------------------------------------------------------\\n# sliding-window runner (same padding / step / Gaussian / accumulation as the predictor)\\n# ---------------------------------------------------------------------------------\\n@torch.inference_mode()\\ndef run_outputs(predictor, data, embeddings, outputs, attn_acc_factory=None, attn_rows=None, on_patch=None):\\n    \\"\\"\\"Sliding-window inference for several *outputs* that share one encoder pass per patch.\\n\\n    data        : preprocessed image tensor (C, X, Y, Z) (from predictor.preprocess)\\n    embeddings  : (1, M, D) text embeddings (rows may be blends built by the caller)\\n    outputs     : list of {\'name\': str, \'row\': int, \'ops\': [...]}; `ops` as in `patched_state`\\n    attn_acc_factory : optional callable(spatial_shape, n_layers, n_rows) -> AttnAccumulator\\n    attn_rows        : accumulate attention only for the first `attn_rows` embedding rows (default: all rows)\\n    on_patch         : optional callable(slicer, states, attns, ctx) called for every patch; `states` = list of\\n                       7 tensors (M, 1, Q) (input state + output of each layer), `attns` = 6 tensors (1, M, S),\\n                       patch_ctx = {\'revert\': padding-revert slicer, \'padded_shape\': spatial shape after padding}\\n    Returns (logits (n_out, X, Y, Z) half, attention (L, M, X, Y, Z) or None), padding reverted.\\"\\"\\"\\n    from acvl_utils.cropping_and_padding.padding import pad_nd_image\\n    from nnunetv2.inference.sliding_window_prediction import compute_gaussian\\n\\n    device = predictor.device\\n    model = predictor.network.to(device)\\n    data_p, revert = pad_nd_image(data, predictor.patch_size, \'constant\', {\'value\': 0}, True, None)\\n    slicers = predictor._internal_get_sliding_window_slicers(data_p.shape[1:])\\n    data_p = data_p.to(device)\\n    emb = embeddings[0].to(device)  # (M, D)\\n\\n    n_out = len(outputs)\\n    logits = torch.zeros((n_out, *data_p.shape[1:]), dtype=torch.half, device=device)\\n    npred = torch.zeros(data_p.shape[1:], dtype=torch.half, device=device)\\n    gaussian = compute_gaussian(tuple(predictor.patch_size), sigma_scale=1. / 8,\\n                                value_scaling_factor=10, device=device)\\n    attn_acc = None\\n    if attn_acc_factory is not None:\\n        n_attn = attn_rows if attn_rows is not None else emb.shape[0]\\n        attn_acc = attn_acc_factory(tuple(data_p.shape[1:]), len(model.transformer_decoder.layers), n_attn)\\n    patch_ctx = {\\"revert\\": revert, \\"padded_shape\\": tuple(data_p.shape[1:])}\\n    ctx = torch.autocast(device.type) if device.type == \'cuda\' else contextlib.nullcontext()\\n\\n    for slicer in slicers:\\n        patch = data_p[slicer][None]\\n        with ctx:\\n            skips, memory = encode(model, patch)\\n            tgt0 = model.project_text_embed(emb).unsqueeze(1)  # (M, 1, Q)\\n            states, attns = base_trajectory(model, tgt0, memory, model.pos_embed)\\n            embs = []\\n            for o in outputs:\\n                r = o[\'row\']\\n                if o.get(\'ops\'):\\n                    t = patched_state(model, tgt0[r:r + 1], memory, model.pos_embed, states, o[\'ops\'])\\n                else:\\n                    t = states[-1][r:r + 1]\\n                embs.append(model.transformer_decoder.norm(t))\\n            mask_emb = torch.cat(embs, dim=0)[:, 0]  # (n_out, Q)\\n            pred = decode(model, skips, mask_emb)\\n        pred = (pred.float() * gaussian.float()).half()\\n        logits[(slice(None), *slicer[1:])] += pred\\n        npred[slicer[1:]] += gaussian.half()\\n        if attn_acc is not None:\\n            stack = torch.stack([a[0] for a in attns]).float()   # (L, M, S)\\n            attn_acc.add(slicer, stack[:, :n_attn], gaussian)\\n        if on_patch is not None:\\n            on_patch(slicer, states, attns, patch_ctx)\\n\\n    torch.div(logits, npred, out=logits)\\n    if torch.any(torch.isinf(logits)):\\n        raise RuntimeError(\'inf in logits\')\\n    logits = logits[(slice(None), *revert[1:])]\\n    attn = attn_acc.finalize(revert) if attn_acc is not None else None\\n    return logits, attn\\n\\n\\n# ---------------------------------------------------------------------------------\\n# scoring (numpy, side-agnostic: uses overlap with the two candidate ground-truth organs)\\n# ---------------------------------------------------------------------------------\\ndef _dice(a, b):\\n    d = a.sum() + b.sum()\\n    return 1.0 if d == 0 else float(2.0 * np.logical_and(a, b).sum() / d)\\n\\n\\ndef score_masks(mask, gt_a, gt_b, cov_thr=0.3, other_thr=0.2):\\n    \\"\\"\\"Overlap of a predicted mask with two candidate ground-truth organs.\\n    cov_x = |mask & gt_x| / |gt_x|. `call` is \'a\' / \'b\' when the mask covers that organ\\n    (>= cov_thr) and the other one less than other_thr, \'both\' when both are covered,\\n    \'empty\' for an empty mask and \'neither\' otherwise.\\"\\"\\"\\n    mask, gt_a, gt_b = (np.asarray(x, dtype=bool) for x in (mask, gt_a, gt_b))\\n    n = int(mask.sum())\\n    cov_a = float(np.logical_and(mask, gt_a).sum() / max(gt_a.sum(), 1))\\n    cov_b = float(np.logical_and(mask, gt_b).sum() / max(gt_b.sum(), 1))\\n    if n == 0:\\n        call = \'empty\'\\n    elif cov_a >= cov_thr and cov_b >= cov_thr:\\n        call = \'both\'\\n    elif cov_a >= cov_thr and cov_b < other_thr:\\n        call = \'a\'\\n    elif cov_b >= cov_thr and cov_a < other_thr:\\n        call = \'b\'\\n    else:\\n        call = \'neither\'\\n    return {\'n_pred_voxels\': n, \'cov_a\': round(cov_a, 4), \'cov_b\': round(cov_b, 4),\\n            \'dice_a\': round(_dice(mask, gt_a), 4), \'dice_b\': round(_dice(mask, gt_b), 4), \'call\': call}\\n"')
_EXP_SRC = json.loads('"\\"\\"\\"\\nExperiments for plan rows 12-16 (all take a VoxTellPredictor-like object and plain numpy arrays, so they can be\\ntested without a GPU).\\n\\n  row 12  run_flip                image left-right flip, prompt unchanged\\n  row 13  run_erasure             organ filled with noise / both organs filled with noise / noise image\\n  row 14  run_interventions       embedding blend and per-layer patching between \\"left X\\" and \\"right X\\"\\n  row 15  (same run)              cross-attention of every prompt over every organ region, per decoder layer\\n  row 16  (same run)              distance between the left and right query states at every decoder layer\\n\\nArrays: `img_arr` is (1, X, Y, Z) in the RAS-reoriented grid used by the notebooks, `label_vol` is (X, Y, Z).\\n`rl_axis` is the right-left axis of the spatial arrays (2 in the notebooks).\\n\\"\\"\\"\\n\\nimport numpy as np\\nimport torch\\n\\nimport voxtell_mechanistic as vm\\n\\n\\n# ------------------------------------------------------------------------------------------------\\ndef mirror(organ):\\n    \\"\\"\\"\'left kidney\' <-> \'right kidney\'.\\"\\"\\"\\n    side, core = organ.split(None, 1)\\n    return (\\"right \\" if side == \\"left\\" else \\"left \\") + core\\n\\n\\ndef centroid(mask):\\n    idx = np.nonzero(mask)\\n    return np.array([c.mean() for c in idx]) if len(idx[0]) else None\\n\\n\\ndef _to_full_masks(logits, bbox, orig_shape):\\n    from acvl_utils.cropping_and_padding.bounding_boxes import insert_crop_into_image\\n    with torch.no_grad():\\n        cropped = torch.cat([(torch.sigmoid(logits[i:i + 8].float()) > 0.5).to(\\"cpu\\", torch.uint8)\\n                             for i in range(0, logits.shape[0], 8)]).numpy()\\n    full = np.zeros((cropped.shape[0], *orig_shape), dtype=np.uint8)\\n    return insert_crop_into_image(full, cropped, bbox).astype(bool)\\n\\n\\ndef predict_masks(predictor, img_arr, prompts, chunk=32):\\n    \\"\\"\\"Same steps as the notebooks: preprocess, embed, sliding window, threshold, put back into the full volume.\\"\\"\\"\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb = predictor.embed_text_prompts(list(prompts))\\n    parts = []\\n    for s in range(0, emb.shape[1], chunk):\\n        parts.append(predictor.predict_sliding_window_return_logits(data, emb[:, s:s + chunk]).to(\\"cpu\\"))\\n        torch.cuda.empty_cache() if torch.cuda.is_available() else None\\n    return _to_full_masks(torch.cat(parts, dim=0), bbox, orig_shape)\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 12 -- image left-right flip\\n# ------------------------------------------------------------------------------------------------\\ndef run_flip(predictor, img_arr, label_vol, organs, ids, prompts, rl_axis=2):\\n    \\"\\"\\"Flip the image and its labels along the right-left axis and keep every prompt unchanged.\\n    call \'a\' = the mask covers the organ that the prompt names (follows the anatomy / the label);\\n    call \'b\' = the mask covers the organ on the other side (follows the side of the image).\\"\\"\\"\\n    flipped_img = np.ascontiguousarray(np.flip(img_arr, axis=rl_axis + 1))\\n    flipped_lab = np.ascontiguousarray(np.flip(label_vol, axis=rl_axis))\\n    rows = []\\n    for cond, img, lab in ((\\"original\\", img_arr, label_vol), (\\"flipped\\", flipped_img, flipped_lab)):\\n        masks = predict_masks(predictor, img, [prompts[o] for o in organs])\\n        for k, o in enumerate(organs):\\n            sc = vm.score_masks(masks[k], lab == ids[o], lab == ids[mirror(o)])\\n            rows.append({\\"organ\\": o, \\"condition\\": cond, \\"prompt\\": prompts[o], \\"n_pred_voxels\\": sc[\\"n_pred_voxels\\"],\\n                         \\"cov_named_organ\\": sc[\\"cov_a\\"], \\"cov_other_side_organ\\": sc[\\"cov_b\\"],\\n                         \\"dice_named_organ\\": sc[\\"dice_a\\"], \\"dice_other_side_organ\\": sc[\\"dice_b\\"],\\n                         \\"call\\": {\\"a\\": \\"follows_named_organ\\", \\"b\\": \\"follows_image_side\\"}.get(sc[\\"call\\"], sc[\\"call\\"])})\\n    return rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 13 -- organ replaced by noise / noise image\\n# ------------------------------------------------------------------------------------------------\\ndef noise_region(img_arr, region, rng):\\n    \\"\\"\\"Replace the voxels of `region` by Gaussian noise with the mean and spread of the image\'s non-zero voxels.\\"\\"\\"\\n    out = img_arr.copy()\\n    vals = img_arr[img_arr != 0]\\n    out[0][region] = rng.normal(float(vals.mean()), float(vals.std()), size=int(region.sum()))\\n    return out\\n\\n\\ndef noise_image(img_arr, rng):\\n    vals = img_arr[img_arr != 0]\\n    return rng.normal(float(vals.mean()), float(vals.std()), size=img_arr.shape).astype(img_arr.dtype)\\n\\n\\ndef _region_metrics(mask, region, orig_voxels):\\n    n = int(mask.sum())\\n    inter = int(np.logical_and(mask, region).sum())\\n    cm, cr = centroid(mask), centroid(region)\\n    return {\\"n_pred_voxels\\": n, \\"size_vs_original\\": round(n / orig_voxels, 4) if orig_voxels else float(\\"nan\\"),\\n            \\"cov_of_noised_region\\": round(inter / max(int(region.sum()), 1), 4),\\n            \\"frac_of_mask_in_region\\": round(inter / n, 4) if n else 0.0,\\n            \\"centroid_offset_vox\\": round(float(np.linalg.norm(cm - cr)), 1) if cm is not None else float(\\"nan\\"),\\n            \\"empty\\": n == 0}\\n\\n\\ndef run_erasure(predictor, img_arr, label_vol, organs, ids, prompts, seed=0):\\n    rng = np.random.default_rng(seed)\\n    regions = {o: label_vol == ids[o] for o in organs}\\n    orig = predict_masks(predictor, img_arr, [prompts[o] for o in organs])\\n    orig_n = {o: int(orig[k].sum()) for k, o in enumerate(organs)}\\n    rows = []\\n\\n    def add(cond, o, mask):\\n        rows.append({\\"organ\\": o, \\"condition\\": cond, \\"prompt\\": prompts[o],\\n                     **_region_metrics(mask, regions[o], orig_n[o])})\\n\\n    for k, o in enumerate(organs):\\n        add(\\"original\\", o, orig[k])\\n    for o in organs:                                        # only the named organ replaced by noise\\n        img = noise_region(img_arr, regions[o], rng)\\n        add(\\"noise_target\\", o, predict_masks(predictor, img, [prompts[o]])[0])\\n    for o in [x for x in organs if x.startswith(\\"left\\") and mirror(x) in organs]:   # both organs of the pair replaced by noise\\n        img = noise_region(noise_region(img_arr, regions[o], rng), regions[mirror(o)], rng)\\n        m = predict_masks(predictor, img, [prompts[o], prompts[mirror(o)]])\\n        add(\\"noise_pair\\", o, m[0])\\n        add(\\"noise_pair\\", mirror(o), m[1])\\n    noisy = predict_masks(predictor, noise_image(img_arr, rng), [prompts[o] for o in organs])\\n    for k, o in enumerate(organs):\\n        add(\\"noise_image\\", o, noisy[k])\\n    return rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# rows 14-16 -- embedding intervention, cross-attention, layer-by-layer distance\\n# ------------------------------------------------------------------------------------------------\\nclass LayerDistanceRecorder:\\n    \\"\\"\\"Row 16: cosine distance and relative L2 distance between two prompts\' query states, per decoder layer\\n    (index 0 = the input state, 1..6 = after each layer), averaged over the patches that contain the organs\\n    and over all patches.\\"\\"\\"\\n\\n    def __init__(self, pair_defs, union_crop):\\n        self.pair_defs, self.union_crop = pair_defs, union_crop      # [(name, row_i, row_j)]\\n        self.union_padded = None\\n        self.cos, self.l2, self.flags = [], [], []\\n\\n    def reset(self):\\n        self.cos, self.l2, self.flags = [], [], []\\n\\n    def on_patch(self, slicer, states, attns, ctx):\\n        if self.union_padded is None:\\n            self.union_padded = np.zeros(ctx[\\"padded_shape\\"], bool)\\n            self.union_padded[ctx[\\"revert\\"][1:]] = self.union_crop\\n        self.flags.append(bool(self.union_padded[slicer[1:]].any()))\\n        c = np.zeros((len(self.pair_defs), len(states)))\\n        d = np.zeros_like(c)\\n        for pi, (_, i, j) in enumerate(self.pair_defs):\\n            for l, st in enumerate(states):\\n                a, b = st[i, 0].float(), st[j, 0].float()\\n                c[pi, l] = 1 - float(torch.nn.functional.cosine_similarity(a, b, dim=0))\\n                d[pi, l] = float(torch.linalg.norm(a - b) / (0.5 * (torch.linalg.norm(a) + torch.linalg.norm(b)) + 1e-8))\\n        self.cos.append(c)\\n        self.l2.append(d)\\n\\n    def rows(self):\\n        cos, l2, flags = np.array(self.cos), np.array(self.l2), np.array(self.flags)\\n        out = []\\n        for scope, sel in ((\\"organ_patches\\", flags), (\\"all_patches\\", np.ones_like(flags))):\\n            if not sel.any():\\n                continue\\n            for pi, (name, _, _) in enumerate(self.pair_defs):\\n                for l in range(cos.shape[2]):\\n                    out.append({\\"pair\\": name, \\"scope\\": scope, \\"layer\\": l, \\"n_patches\\": int(sel.sum()),\\n                                \\"cosine_distance\\": round(float(cos[sel, pi, l].mean()), 6),\\n                                \\"relative_l2_distance\\": round(float(l2[sel, pi, l].mean()), 6)})\\n        return out\\n\\n\\ndef _attention_rows(attn, organs, regions_crop, rl_axis):\\n    \\"\\"\\"Row 15. attn: (L, n_prompts, X, Y, Z) attention density (mean 1) in crop space; regions_crop: {organ: bool}.\\"\\"\\"\\n    dens_shape = attn.shape[2:]\\n    coords = torch.arange(dens_shape[rl_axis], dtype=torch.float32, device=attn.device)\\n    view = [1, 1, 1]\\n    view[rl_axis] = -1\\n    coords = coords.view(*view)\\n    rows, com_rows = [], []\\n    for l in range(attn.shape[0]):\\n        for p, prompt_organ in enumerate(organs):\\n            d = attn[l, p].float()\\n            total, mean_all = d.sum(), d.mean()\\n            for region_organ in organs:\\n                r = torch.from_numpy(regions_crop[region_organ]).to(d.device)\\n                if not bool(r.any()):\\n                    continue\\n                rows.append({\\"layer\\": l + 1, \\"prompt_organ\\": prompt_organ, \\"region_organ\\": region_organ,\\n                             \\"is_matched\\": region_organ == prompt_organ, \\"is_wrong_side\\": region_organ == mirror(prompt_organ),\\n                             \\"enrichment\\": round(float(d[r].mean() / mean_all), 4),\\n                             \\"mass_fraction\\": round(float(d[r].sum() / total), 6)})\\n            com = float((d * coords).sum() / total)\\n            named = centroid(regions_crop[prompt_organ])\\n            other = centroid(regions_crop[mirror(prompt_organ)]) if mirror(prompt_organ) in regions_crop else None\\n            if named is not None and other is not None:\\n                com_rows.append({\\"layer\\": l + 1, \\"prompt_organ\\": prompt_organ, \\"attention_centre_rl\\": round(com, 2),\\n                                 \\"named_organ_centre_rl\\": round(float(named[rl_axis]), 2),\\n                                 \\"other_side_organ_centre_rl\\": round(float(other[rl_axis]), 2),\\n                                 \\"closer_to_named_organ\\": abs(com - named[rl_axis]) < abs(com - other[rl_axis])})\\n    return rows, com_rows\\n\\n\\n_OOM = getattr(torch, \\"OutOfMemoryError\\", torch.cuda.OutOfMemoryError)      # older torch only has the cuda one\\n\\n\\ndef _run_chunked(predictor, data, emb_all, outputs, bbox, orig_shape, attn_factory, attn_rows, on_patch, reset, chunk=12):\\n    \\"\\"\\"Run the (many) outputs in chunks so the logits / decoder activations fit on the GPU. The attention and the layer\\n    recorder only need one pass, so they ride on the first chunk. A chunk that runs out of memory is retried with half the\\n    size. Returns (bool masks (n_out, X, Y, Z) in the original grid, attention or None).\\"\\"\\"\\n    import gc\\n    masks, attn, i = [], None, 0\\n    while i < len(outputs):\\n        part, first = outputs[i:i + chunk], i == 0\\n        oom = False\\n        try:\\n            logits, a = vm.run_outputs(predictor, data, emb_all, part,\\n                                       attn_acc_factory=attn_factory if first else None,\\n                                       attn_rows=attn_rows if first else None, on_patch=on_patch if first else None)\\n        except _OOM:\\n            oom = True\\n        if oom:                                   # clean up outside the except block so the failed frames are released\\n            gc.collect(); torch.cuda.empty_cache()\\n            if chunk == 1:\\n                raise RuntimeError(\\"out of GPU memory even with one output per pass\\")\\n            chunk = max(1, chunk // 2)\\n            if first:\\n                reset()\\n            continue\\n        masks.append(_to_full_masks(logits, bbox, orig_shape))\\n        del logits\\n        if first:\\n            attn = a.cpu() if a is not None else None\\n        i += len(part)\\n        gc.collect(); torch.cuda.empty_cache()\\n    return np.concatenate(masks), attn\\n\\n\\ndef run_interventions(predictor, img_arr, label_vol, organs, ids, prompts, rl_axis=2,\\n                      alphas=(-0.5, 0.25, 0.5, 0.75, 1.5), single_layers=range(6), from_layers=(0, 3),\\n                      state_layers=(0, 5), map_downsample=4, out_chunk=12):\\n    \\"\\"\\"Returns dict(row14=[...], row15=[...], row15_centre=[...], row16=[...], attention_maps={name: float16 array}).\\"\\"\\"\\n    pairs = [(o, mirror(o)) for o in organs if o.startswith(\\"left\\") and mirror(o) in organs]\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb_base = predictor.embed_text_prompts([prompts[o] for o in organs])          # (1, n, D)\\n    row_of = {o: i for i, o in enumerate(organs)}\\n    extra = []\\n    blend_row = {}\\n    for L, R in pairs:\\n        for a in alphas:\\n            blend_row[(L, a)] = len(organs) + len(extra)\\n            extra.append((1 - a) * emb_base[0, row_of[L]] + a * emb_base[0, row_of[R]])\\n    emb_all = torch.cat([emb_base, torch.stack(extra)[None]], dim=1) if extra else emb_base\\n\\n    outputs, meta = [], []\\n\\n    def add(name, row, ops, m):\\n        outputs.append({\\"name\\": name, \\"row\\": row, \\"ops\\": ops})\\n        meta.append(m)\\n\\n    for L, R in pairs:\\n        core = L.split(None, 1)[1]\\n        for o in (L, R):\\n            add(f\\"{o}|baseline\\", row_of[o], [], dict(pair=core, variant=\\"baseline\\", dst=o.split()[0], src=\\"\\", alpha=np.nan, layer=-1))\\n        for a in alphas:\\n            add(f\\"{L}|blend|{a}\\", blend_row[(L, a)], [], dict(pair=core, variant=\\"blend\\", dst=\\"left\\", src=\\"right\\", alpha=a, layer=-1))\\n        for dst, src in ((L, R), (R, L)):\\n            d, s = dst.split()[0], src.split()[0]\\n            for k in single_layers:\\n                add(f\\"{dst}|update|{k}\\", row_of[dst], [{\\"layer\\": k, \\"mode\\": \\"update\\", \\"src\\": row_of[src]}],\\n                    dict(pair=core, variant=\\"single_layer_update\\", dst=d, src=s, alpha=np.nan, layer=k))\\n            for k in from_layers:\\n                add(f\\"{dst}|from|{k}\\", row_of[dst], [{\\"layer\\": j, \\"mode\\": \\"update\\", \\"src\\": row_of[src]} for j in range(k, 6)],\\n                    dict(pair=core, variant=\\"updates_from_layer\\", dst=d, src=s, alpha=np.nan, layer=k))\\n            for k in state_layers:\\n                add(f\\"{dst}|state|{k}\\", row_of[dst], [{\\"layer\\": k, \\"mode\\": \\"state\\", \\"src\\": row_of[src]}],\\n                    dict(pair=core, variant=\\"whole_state_control\\", dst=d, src=s, alpha=np.nan, layer=k))\\n\\n    # ground truth in crop space (for the attention statistics and the recorder)\\n    crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)\\n    label_crop = label_vol[crop_sl]\\n    regions_crop = {o: label_crop == ids[o] for o in organs}\\n    union_crop = np.zeros(label_crop.shape, bool)\\n    for r in regions_crop.values():\\n        union_crop |= r\\n    ref_pairs = []\\n    for L, R in pairs:\\n        others = [o for o in organs if o.split(None, 1)[1] != L.split(None, 1)[1] and o.startswith(\\"left\\")]\\n        ref_pairs.append((f\\"{L.split(None, 1)[1]}: left vs right\\", row_of[L], row_of[R]))\\n        if others:\\n            ref_pairs.append((f\\"{L.split(None, 1)[1]}: {L} vs {others[0]} (reference)\\", row_of[L], row_of[others[0]]))\\n    recorder = LayerDistanceRecorder(ref_pairs, union_crop)\\n\\n    masks, attn = _run_chunked(\\n        predictor, data, emb_all, outputs, bbox, orig_shape,\\n        attn_factory=lambda shape, n_layers, n_rows: vm.AttnAccumulator(shape, n_layers, n_rows, predictor.patch_size, predictor.device),\\n        attn_rows=len(organs), on_patch=recorder.on_patch, reset=recorder.reset, chunk=out_chunk)\\n    base_idx = {(m[\\"pair\\"], m[\\"dst\\"]): k for k, m in enumerate(meta) if m[\\"variant\\"] == \\"baseline\\"}\\n    rows14 = []\\n    for k, m in enumerate(meta):\\n        L = f\\"left {m[\'pair\']}\\"\\n        R = f\\"right {m[\'pair\']}\\"\\n        sc = vm.score_masks(masks[k], label_vol == ids[L], label_vol == ids[R])\\n        call = {\\"a\\": \\"left\\", \\"b\\": \\"right\\"}.get(sc[\\"call\\"], sc[\\"call\\"])\\n        moved = bool((m[\\"dst\\"] == \\"left\\" and call == \\"right\\") or (m[\\"dst\\"] == \\"right\\" and call == \\"left\\")) if m[\\"variant\\"] != \\"blend\\" else np.nan\\n        agree = float((masks[k] == masks[base_idx[(m[\\"pair\\"], m[\\"src\\"])]]).mean()) if m[\\"variant\\"] == \\"whole_state_control\\" else np.nan\\n        rows14.append({**m, \\"agreement_with_source_baseline\\": round(agree, 6) if agree == agree else agree,\\n                       \\"n_pred_voxels\\": sc[\\"n_pred_voxels\\"], \\"cov_left_organ\\": sc[\\"cov_a\\"], \\"cov_right_organ\\": sc[\\"cov_b\\"],\\n                       \\"dice_left_organ\\": sc[\\"dice_a\\"], \\"dice_right_organ\\": sc[\\"dice_b\\"], \\"call\\": call, \\"moved_to_source_side\\": moved})\\n    rows15, com15 = _attention_rows(attn, organs, regions_crop, rl_axis)\\n    maps = {}\\n    if map_downsample:\\n        pool = torch.nn.AvgPool3d(map_downsample, ceil_mode=True)\\n        for p, o in enumerate(organs):\\n            last = pool(attn[-1, p][None, None].float())[0, 0]\\n            mean = pool(attn[:, p].float().mean(0)[None, None])[0, 0]\\n            maps[f\\"{o}__last_layer\\"] = last.half().cpu().numpy()\\n            maps[f\\"{o}__mean_of_layers\\"] = mean.half().cpu().numpy()\\n    return {\\"row14\\": rows14, \\"row15\\": rows15, \\"row15_centre\\": com15, \\"row16\\": recorder.rows(), \\"attention_maps\\": maps}\\n\\n\\ndef validate_custom_loop(predictor, img_arr, prompts):\\n    \\"\\"\\"The custom sliding-window loop must reproduce the predictor\'s own masks; returns the voxel agreement.\\"\\"\\"\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb = predictor.embed_text_prompts(list(prompts))\\n    ref = predictor.predict_sliding_window_return_logits(data, emb)\\n    mine, _ = vm.run_outputs(predictor, data, emb, [{\\"name\\": str(i), \\"row\\": i, \\"ops\\": []} for i in range(emb.shape[1])])\\n    return float(((ref.float() > 0) == (mine.float() > 0)).float().mean())\\n"')
_m = types.ModuleType('voxtell_mechanistic')
exec(compile(_MECH_SRC, 'voxtell_mechanistic', 'exec'), _m.__dict__)
sys.modules['voxtell_mechanistic'] = _m
_e = types.ModuleType('voxtell_experiments')
exec(compile(_EXP_SRC, 'voxtell_experiments', 'exec'), _e.__dict__)
sys.modules['voxtell_experiments'] = _e
import voxtell_mechanistic as vm
import voxtell_experiments as ve
print('experiment modules loaded')


## 2. Subjects, organs and helpers

In [ ]:
import gc

DATASET_NAME = 'neurite_oasis'
SUBJECTS = [
    ("faria", "OASIS_OAS1_0014_MR1"),
    ("faria", "OASIS_OAS1_0057_MR1"),
    ("hridoy", "OASIS_OAS1_0123_MR1"),
    ("hridoy", "OASIS_OAS1_0135_MR1"),
    ("nipu", "OASIS_OAS1_0062_MR1"),
    ("faria", "OASIS_OAS1_0363_MR1"),
    ("hridoy", "OASIS_OAS1_0419_MR1"),
    ("nipu", "OASIS_OAS1_0076_MR1"),
    ("nipu", "OASIS_OAS1_0151_MR1"),
    ("nipu", "OASIS_OAS1_0421_MR1")
]
ORGAN_CORES = ['cerebral white matter', 'cerebellum cortex']   # None = every bilateral organ

subject_organs = {}
for person, sid in SUBJECTS:
    rows = [r for r in load_manifest_rot000(person, DATASET_NAME) if r['subject_id'] == sid]
    rows = [r for r in rows if r['organ'].split()[0].lower() in ('left', 'right')]
    if ORGAN_CORES:
        rows = [r for r in rows if r['organ'].split(None, 1)[1] in ORGAN_CORES]
    have = {r['organ'] for r in rows}
    rows = [r for r in rows if ve.mirror(r['organ']) in have]          # keep complete left/right pairs only
    assert rows, f'no bilateral organs for {person}/{sid}'
    subject_organs[sid] = {'person': person, 'rows': rows}
print(len(subject_organs), 'subjects;', 'organs per subject:', [len(v['rows']) for v in subject_organs.values()][:3], '...')


def load_subject(sid):
    info = subject_organs[sid]
    rows = info['rows']
    base = RESEARCH_REPO / info['person'] / 'samples' / DATASET_NAME
    img_arr, _ = load_volume_ras(base / rows[0]['image_path'])
    label_vol, _ = load_seg_ras(base / rows[0]['label_path'])
    organs = [r['organ'] for r in rows]
    ids = {r['organ']: int(r['label_id']) for r in rows}
    prompts = {r['organ']: r['prompt_text'] for r in rows}
    return info['person'], img_arr, label_vol, organs, ids, prompts


def append_csv(path, rows, first):
    if not rows:
        return
    write_header = first or not path.exists()      # also right if an earlier subject produced no rows for this file
    with open(path, 'w' if first else 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        if write_header:
            w.writeheader()
        w.writerows(rows)


def tag(rows, sid, person):
    return [{'dataset': DATASET_NAME, 'person': person, 'subject_id': sid, **r} for r in rows]


## 3. Check: the custom sliding-window loop must match the predictor

In [ ]:
# The custom loop must reproduce the predictor's own masks, otherwise rows 14-16 would not be trustworthy.
_sid = next(iter(subject_organs))
_, _img, _lab, _organs, _ids, _prompts = load_subject(_sid)
_agree = ve.validate_custom_loop(predictor, _img, [_prompts[o] for o in _organs])
print(f'custom sliding-window loop vs. the predictor: {_agree * 100:.3f}% of voxels have the same sign')
assert _agree > 0.995, 'the custom loop does not match the predictor -- do not trust rows 14-16'
del _img, _lab
gc.collect(); torch.cuda.empty_cache()


## 4. Rows 12 and 13 -- image flip, organ noise, noise image

Public-API predictions only (no changes inside the model). Row 12 flips the image and its labels along the right-left axis and keeps every prompt. Row 13 replaces the named organ (and, separately, both organs of the pair) with Gaussian noise, or replaces the whole image by noise with the image's mean and spread.

In [ ]:
import traceback

r12_path = RESULTS_ROOT / f'row12_image_flip_neurite_oasis.csv'
r13_path = RESULTS_ROOT / f'row13_organ_noise_neurite_oasis.csv'
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        rows12 = ve.run_flip(predictor, img_arr, label_vol, organs, ids, prompts, rl_axis=RL_AXIS)
        rows13 = ve.run_erasure(predictor, img_arr, label_vol, organs, ids, prompts, seed=si)
        append_csv(r12_path, tag(rows12, sid, person), first)
        append_csv(r13_path, tag(rows13, sid, person), first)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: rows 12-13 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r12_path.name, r13_path.name)


## 5. Rows 14, 15 and 16 -- one custom pass per subject

Per subject and left/right pair: baselines, five embedding blends (alpha = -0.5, 0.25, 0.5, 0.75, 1.5 between "left" and "right"), single-layer patches of the query update (all six layers, both directions), patches from a layer onward, and whole-state controls -- while recording the cross-attention (row 15) and the left/right distance per layer (row 16).

In [ ]:
import numpy as np, traceback

(RESULTS_ROOT / 'attention_maps').mkdir(exist_ok=True)
paths = {k: RESULTS_ROOT / f'{k}_neurite_oasis.csv' for k in
         ('row14_embedding_intervention', 'row15_attention_enrichment', 'row15_attention_centre', 'row16_layer_distance')}
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        res = ve.run_interventions(predictor, img_arr, label_vol, organs, ids, prompts, rl_axis=RL_AXIS)
        append_csv(paths['row14_embedding_intervention'], tag(res['row14'], sid, person), first)
        append_csv(paths['row15_attention_enrichment'], tag(res['row15'], sid, person), first)
        append_csv(paths['row15_attention_centre'], tag(res['row15_centre'], sid, person), first)
        append_csv(paths['row16_layer_distance'], tag(res['row16'], sid, person), first)
        np.savez_compressed(RESULTS_ROOT / 'attention_maps' / f'{sid}.npz', **res['attention_maps'])
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: rows 14-16 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', [p.name for p in paths.values()])


## 6. Summary and archive

In [ ]:
import shutil
import pandas as pd

pd.set_option('display.width', 200, 'display.max_columns', 30)
R = lambda n: pd.read_csv(RESULTS_ROOT / f'{n}_neurite_oasis.csv')

print('=== Row 12: image flip -- what does the mask cover? (share of organs) ===')
d = R('row12_image_flip')
print(d.groupby(['condition', 'call']).size().unstack(fill_value=0))

print('\n=== Row 13: organ noise -- mask size vs. the original prediction, share of empty masks ===')
d = R('row13_organ_noise')
print(d.groupby('condition').agg(size_vs_original=('size_vs_original', 'median'), empty=('empty', 'mean'),
                                 in_noised_region=('frac_of_mask_in_region', 'mean')).round(3))

print('\n=== Row 14: which side does the mask land on? ===')
d = R('row14_embedding_intervention')
b = d[d.variant == 'blend']
print('blend (alpha 0 = "left", 1 = "right"):'); print(b.groupby(['alpha', 'call']).size().unstack(fill_value=0))
u = d[d.variant == 'single_layer_update']
print('single-layer update from the other prompt -- share of masks that moved to the source side, by layer:')
print(u.groupby(['dst', 'layer']).moved_to_source_side.mean().unstack().round(2))
c = d[d.variant == 'whole_state_control']
print('positive control (a whole-state patch must reproduce the source mask): voxel agreement with the source prompt, mean / min:',
      c.agreement_with_source_baseline.mean().round(5), '/', c.agreement_with_source_baseline.min().round(5), '(expected >= 0.999)')

print('\n=== Row 15: attention enrichment (1 = as much as elsewhere), by layer ===')
a = R('row15_attention_enrichment')
a['kind'] = np.where(a.is_matched, 'named organ', np.where(a.is_wrong_side, 'other side, same organ', 'different organ'))
print(a.groupby(['layer', 'kind']).enrichment.mean().unstack().round(2))
cc = R('row15_attention_centre')
print('attention centre closer to the named organ (share):'); print(cc.groupby('layer').closer_to_named_organ.mean().round(2).to_dict())

print('\n=== Row 16: cosine distance between the query states, by layer (0 = input) ===')
l = R('row16_layer_distance')
print(l[l.scope == 'organ_patches'].pivot_table(index='layer', columns='pair', values='cosine_distance').round(3))

zip_path = WORKING_DIR / 'voxtell_neurite_oasis_mechanistic_results'
shutil.make_archive(str(zip_path), 'zip', str(RESULTS_ROOT))
print('\nArchive:', str(zip_path) + '.zip')


## Outputs (all under `results/`)

| File | Row | Content |
|---|---|---|
| `row12_image_flip_neurite_oasis.csv` | 12 | per organ, original vs. flipped: coverage of the named organ / the organ on the other side, `call` = `follows_named_organ` or `follows_image_side` |
| `row13_organ_noise_neurite_oasis.csv` | 13 | per organ and condition (`original`, `noise_target`, `noise_pair`, `noise_image`): mask size vs. the original prediction, coverage of the noised region, offset of the mask centre |
| `row14_embedding_intervention_neurite_oasis.csv` | 14 | per pair: baselines, embedding blends (alpha), single-layer update patches, updates from a layer onward, whole-state control; side of the resulting mask |
| `row15_attention_enrichment_neurite_oasis.csv` | 15 | per prompt x organ region x decoder layer: attention enrichment and mass fraction |
| `row15_attention_centre_neurite_oasis.csv` | 15 | per prompt x layer: where the attention centre lies vs. the named and the other-side organ |
| `row16_layer_distance_neurite_oasis.csv` | 16 | cosine / relative L2 distance between the left and right query states per layer (layer 0 = input), with an organ-vs-organ reference |
| `attention_maps/<subject>.npz` | 15 | down-sampled attention maps (last layer and mean of layers) for plotting |

Reading guide: in row 14, `moved_to_source_side` is True when patching with the *other* prompt's contribution turns the mask
into the other side. `agreement_with_source_baseline` is the machinery check: for every `whole_state_control` row it should be
about 1 (>= 0.999; patching a whole layer state must reproduce the source prompt's mask, up to floating-point rounding).
